<a href="https://colab.research.google.com/github/Drewron0/DSA-Bootcamp-Java/blob/main/SATRIA_BDC2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [1] **Data Import**

In [1]:
# Init, import training dataset

import os
from google.colab import drive

drive.mount('/content/drive')

# Path to the shortcut in your Drive
dataset_dir = '/content/drive/MyDrive/BDC2026/BDC2026'

# Verify path
print(f"Dataset exists: {os.path.exists(dataset_dir)}")

Mounted at /content/drive
Dataset exists: True


In [2]:
# Counts the total of training data to verify if data has been imported correctly

train_dir = os.path.join(dataset_dir, "train")
test_dir = os.path.join(dataset_dir, "test")

train_counts = {
    cls: len(os.listdir(os.path.join(train_dir, cls)))
    for cls in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, cls))
}
test_count = len([f for f in os.listdir(test_dir) if os.path.isfile(os.path.join(test_dir, f))])

print(f"Train counts: {train_counts}")
print(f"Total Train: {sum(train_counts.values())} (Expected: 26527)")
print(f"Total Test: {test_count} (Expected: 1458)")

Train counts: {'0_Recyclable': 9999, '1_Electronic': 3961, '2_Organic': 12567}
Total Train: 26527 (Expected: 26527)
Total Test: 1458 (Expected: 1458)


# [2] **Data Preprocessing**

In [3]:
# Setup environment dependencies and global configurations
!pip install timm albumentations scikit-learn -q

import random
import numpy as np
import pandas as pd
import torch

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

SEED = 42
seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset_dir = '/content/drive/MyDrive/BDC2026/BDC2026'

In [4]:
# Construct training dataframe mapping paths to labels
import glob

train_dir = os.path.join(dataset_dir, "train")
class_names = sorted(os.listdir(train_dir))
class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}

data = []
for cls in class_names:
    cls_folder = os.path.join(train_dir, cls)
    if os.path.isdir(cls_folder):
        img_paths = glob.glob(os.path.join(cls_folder, "*"))
        for p in img_paths:
            data.append({"filepath": p, "label": class_to_idx[cls]})

df = pd.DataFrame(data)
df.head()

,filepath,label
0,/content/drive/MyDrive/BDC2026/BDC2026/train/0...,0
1,/content/drive/MyDrive/BDC2026/BDC2026/train/0...,0
2,/content/drive/MyDrive/BDC2026/BDC2026/train/0...,0
3,/content/drive/MyDrive/BDC2026/BDC2026/train/0...,0
4,/content/drive/MyDrive/BDC2026/BDC2026/train/0...,0


In [5]:
# Generate Stratified 5-Fold split
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
df['fold'] = -1

for fold, (_, val_idx) in enumerate(skf.split(df, df['label'])):
    df.loc[val_idx, 'fold'] = fold

df.to_csv("train_folds.csv", index=False)
df.groupby(['fold', 'label']).size().unstack()

label,0,1,2
fold,,,
0,2000,792,2514
1,2000,792,2514
2,2000,792,2513
3,2000,792,2513
4,1999,793,2513


In [6]:
# PyTorch Dataset and Albumentations augmentations
import cv2
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

class WasteDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row['filepath'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        label = torch.tensor(row['label'], dtype=torch.long)
        return image, label

train_transforms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.1, contrast=0.1, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

In [7]:
# Verify dataset reading, augmentation output shapes, and dataloader batching
from torch.utils.data import DataLoader

test_dataset = WasteDataset(df=df, transform=train_transforms)
sample_img, sample_label = test_dataset[0]

print(f"Sample tensor shape : {sample_img.shape}")
print(f"Sample tensor dtype : {sample_img.dtype}")
print(f"Sample label        : {sample_label.item()}")

test_loader = DataLoader(test_dataset, batch_size=16, shuffle=True, num_workers=2)
batch_imgs, batch_labels = next(iter(test_loader))

print(f"Batch images shape  : {batch_imgs.shape}")
print(f"Batch labels shape  : {batch_labels.shape}")

Sample tensor shape : torch.Size([3, 224, 224])
Sample tensor dtype : torch.float32
Sample label        : 0
Batch images shape  : torch.Size([16, 3, 224, 224])
Batch labels shape  : torch.Size([16])


# [3] **Baseline Training Pipeline**

In [8]:
# Define baseline model architecture and class-weighted loss
import timm
import torch.nn as nn

class WasteClassifier(nn.Module):
    def __init__(self, model_name='resnet34', num_classes=3, pretrained=True):
        super().__init__()
        self.model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

# Calculate class weights for imbalance handling
class_counts = df['label'].value_counts().sort_index().values
total_samples = len(df)
num_classes = len(class_counts)

class_weights = total_samples / (num_classes * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)

In [9]:
# Training and validation engine evaluating Macro F1-Score
from sklearn.metrics import f1_score
from tqdm.notebook import tqdm

def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(dataloader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
    return running_loss / len(dataloader.dataset)

@torch.no_grad()
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_targets = []

    for images, labels in tqdm(dataloader, desc="Validating", leave=False):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(labels.cpu().numpy())

    val_loss = running_loss / len(dataloader.dataset)
    macro_f1 = f1_score(all_targets, all_preds, average='macro')
    return val_loss, macro_f1

In [10]:
# Run baseline training on Fold 0
train_df = df[df['fold'] != 0].reset_index(drop=True)
val_df = df[df['fold'] == 0].reset_index(drop=True)

train_ds = WasteDataset(train_df, transform=train_transforms)
val_ds = WasteDataset(val_df, transform=val_transforms)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

model = WasteClassifier('resnet34', pretrained=True).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)

EPOCHS = 5
best_f1 = 0.0

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_f1 = validate(model, val_loader, criterion, DEVICE)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Macro F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), "baseline_resnet34_fold0.pth")

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

Training:   0%|          | 0/664 [00:00<?, ?it/s]

Validating:   0%|          | 0/166 [00:00<?, ?it/s]

Epoch 1/5 | Train Loss: 0.4388 | Val Loss: 0.1788 | Val Macro F1: 0.9391


...

Validating:   0%|          | 0/166 [00:00<?, ?it/s]

Epoch 4/5 | Train Loss: 0.1468 | Val Loss: 0.1295 | Val Macro F1: 0.9524


Training:   0%|          | 0/664 [00:00<?, ?it/s]

Validating:   0%|          | 0/166 [00:00<?, ?it/s]

Epoch 5/5 | Train Loss: 0.1257 | Val Loss: 0.1229 | Val Macro F1: 0.9608


# [4] **Inference & File Submission Generation**

In [11]:
# Setup test dataset and natural numerical sorting for test files
import re

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

class WasteTestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.filenames = sorted(os.listdir(test_dir), key=natural_sort_key)
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        filepath = os.path.join(self.test_dir, filename)

        image = cv2.imread(filepath)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, filename

In [12]:
# Run model inference on test images
test_dir = os.path.join(dataset_dir, "test")
test_ds = WasteTestDataset(test_dir, transform=val_transforms)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

model = WasteClassifier('resnet34', pretrained=False).to(DEVICE)
model.load_state_dict(torch.load("baseline_resnet34_fold0.pth"))
model.eval()

all_preds = []
all_filenames = []

with torch.no_grad():
    for images, filenames in tqdm(test_loader, desc="Predicting Test Data"):
        images = images.to(DEVICE)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_filenames.extend(filenames)

Predicting Test Data:   0%|          | 0/46 [00:00<?, ?it/s]

In [14]:
# Export submission CSV and verify formatting compliance
submission_df = pd.DataFrame({
    'id': range(1, len(all_preds) + 1),
    'predicted': all_preds
})

team_name = "YourTeamName"
output_filename = f"submission_{team_name}.csv"
submission_df.to_csv(output_filename, index=False)

print(f"Saved submission to {output_filename}")
print(f"Row count: {len(submission_df)} (Expected: 1458)")
print(f"Class distribution in predictions:\n{submission_df['predicted'].value_counts()}")
submission_df.head(10)

Saved submission to submission_YourTeamName.csv
Row count: 1458 (Expected: 1458)
Class distribution in predictions:
predicted
2    780
0    461
1    217
Name: count, dtype: int64


,id,predicted
0,1,2
1,2,2
2,3,2
3,4,1
4,5,0
5,6,2
6,7,2
7,8,0
8,9,1
9,10,2


In [16]:
# Verify submission file compliance against competition rules
import pandas as pd

sub = pd.read_csv("submission_YourTeamName.csv")

assert len(sub) == 1458, f"Error: Expected 1458 rows, got {len(sub)}"
assert list(sub.columns) == ['id', 'predicted'], f"Error: Columns must be ['id', 'predicted'], got {list(sub.columns)}"
assert sub['id'].iloc[0] == 1 and sub['id'].iloc[-1] == 1458, "Error: IDs must start at 1 and end at 1458 strictly"
assert sub['predicted'].isnull().sum() == 0, "Error: Found NaN values in predictions"
assert set(sub['predicted'].unique()).issubset({0, 1, 2}), "Error: Invalid class label detected"

print("Check passed")

Check passed
